# Fabric Deployment Accelerator Notebook

Use this notebook from the **0000_deployment accelerator** workspace to execute the deployment accelerator directly inside Microsoft Fabric. The notebook installs dependencies, reads the shared `config/blueprint.yaml`, and invokes the Python deployer to apply the SQL package and regenerate environment manifests.

## 1. Configure paths and options
Adjust the values below to match the location of the repository and the deployment behaviour you require. The defaults assume the repository is synced to `Files/fabric_blueprint_deployment_accelerator` in the workspace and that the orchestration SQL database lives in the workspace **0000 [ Company ] Dataplatform Management & Orchestration**.

In [ ]:
from pathlib import Path

# Update this if your repository is stored in a different Files location
REPO_ROOT = Path('/lakehouse/default/Files/fabric_blueprint_deployment_accelerator')
CONFIG_PATH = REPO_ROOT / 'config' / 'blueprint.yaml'
OUTPUT_ROOT = REPO_ROOT / 'environments'

APPLY_SQL = True          # Set to False if the SQL objects already exist
SEED_DOMAINS = False      # Set to True to push entries from seedDomains into SQL
TARGET_ENVIRONMENTS = []  # Example: ['dev', 'test'] to limit manifest generation
RUN_OFFLINE = False       # True skips SQL connectivity and relies on seedDomains
VERBOSE = True            # Enable detailed logging from the deployer


## 2. Install dependencies (first run per session)

In [ ]:
import sys
import subprocess

requirements = REPO_ROOT / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
else:
    print('requirements.txt not found, skipping pip install')


## 3. Execute the deployment accelerator

In [ ]:
import shlex
import subprocess
import sys

args = [
    sys.executable,
    str(REPO_ROOT / 'setup' / 'fabric_blueprint_deployer.py'),
    '--config', str(CONFIG_PATH),
    '--output-root', str(OUTPUT_ROOT),
]

if APPLY_SQL:
    args.append('--apply-sql')
if SEED_DOMAINS:
    args.append('--seed')
if TARGET_ENVIRONMENTS:
    args.append('--environments')
    args.extend(TARGET_ENVIRONMENTS)
if RUN_OFFLINE:
    args.append('--offline')
if VERBOSE:
    args.append('--verbose')

print('Running deployer with command:')
print(' \n'.join(shlex.quote(str(part)) for part in args))

result = subprocess.run(args, check=False, text=True)
print('Deployer exited with return code', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Deployer execution failed')


## 4. Preview generated manifests (optional)

In [ ]:
import json

def preview_manifest(env: str):
    manifest_path = OUTPUT_ROOT / env / 'manifest.json'
    if not manifest_path.exists():
        print(f'No manifest found for environment {env} at {manifest_path}')
        return
    data = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(f"Manifest summary for {env}:")
    for domain in data.get('domains', []):
        print(f"- {domain['displayName']} ({domain['code']}) -> {len(domain.get('layers', []))} layers")
        for layer in domain.get('layers', []):
            table_count = len(layer.get('tables', []))
            print(f"    · {layer['code']} ({layer['workspaceRole']}) — tables: {table_count}")

# Example usage: preview_manifest('dev')
